# SKM-TEA (qDESS) → nnU-Net v2 converter

Chuyển SKM-TEA sang định dạng nnU-Net:
- **Ảnh**: đọc `echo1` + `echo2` trong `image_files/*.h5` → **RMS** thành 1 kênh giống DESS.
- **Nhãn**: dùng sẵn `segmentation_masks/dicom-track/*.nii.gz`, **remap** về bộ nhãn UNION của dự án.
- **Geometry**: lấy affine từ chính file mask (mask được tạo trên grid này) → đảm bảo ảnh ↔ nhãn khớp tuyệt đối.

**Chạy theo thứ tự.** Cell *Inspect* và *QC overlay* là bắt buộc phải xem trước khi convert hàng loạt.


In [ ]:
!pip install -q nibabel h5py tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0) Cấu hình đường dẫn & ánh xạ nhãn


In [ ]:
from pathlib import Path

SKM_ROOT = Path("/content/drive/MyDrive/SKM-TEA")
IMG_DIR  = SKM_ROOT / "image_files"
# QUAN TRONG: image_files (.h5) = anh RAW-DATA track -> PHAI ghep voi mask raw-data-track.
#   (dicom-track chi khop voi folder dicom/, KHONG khop voi .h5 -> mask se lech!)
MASK_DIR = SKM_ROOT / "segmentation_masks" / "raw-data-track"
ANN_DIR  = SKM_ROOT / "annotations" / "v1.0.0"

# --- Nơi xuất dataset nnU-Net (để trên Drive cho khỏi mất khi Colab reset) ---
DATASET_ID   = 11
DATASET_NAME = f"Dataset{DATASET_ID:03d}_SKMTEA"
OUT_ROOT  = Path("/content/drive/MyDrive/nnUNet_raw") / DATASET_NAME
IMAGES_TR = OUT_ROOT / "imagesTr"
LABELS_TR = OUT_ROOT / "labelsTr"
for d in (IMAGES_TR, LABELS_TR):
    d.mkdir(parents=True, exist_ok=True)

CHANNEL_NAME = "MRI"   # TRÙNG với dataset OAI-ZIB để sau merge cùng 1 channel

# --- Hieu chinh huong anh h5 cho khop mask (dat SAU khi chay cell "Hieu chinh huong") ---
ORIENT_K    = 0       # so lan xoay 90 do trong mat phang (0,1,2,3)
ORIENT_FLIP = False   # lat truc 0 sau khi xoay

# ============================================================================
# ÁNH XẠ LABEL: nhãn gốc SKM-TEA -> bộ nhãn UNION của dự án.
# !!! PHẢI xác nhận bằng cell Inspect (số nhãn) + cell sanity-check (medial/lateral)
#     TRƯỚC KHI tin mapping này. Đặt giá trị = 0 để BỎ class.
#
# Giả định mặc định = SKM-TEA bản 6-class, thứ tự:
#   1 Patellar, 2 Femoral, 3 Med Tibial, 4 Lat Tibial, 5 Med Meniscus, 6 Lat Meniscus
# ============================================================================
LABEL_MAP = {
    1: 8,  # Patellar Cartilage        -> patellar_cartilage (8)   [đặt 0 nếu không cần patella]
    2: 2,  # Femoral Cartilage         -> femoral_cartilage
    3: 4,  # Medial  Tibial Cartilage  -> medial_tibial_cartilage    << VERIFY med/lat
    4: 5,  # Lateral Tibial Cartilage  -> lateral_tibial_cartilage   << VERIFY med/lat
    5: 6,  # Medial  Meniscus          -> medial_meniscus            << VERIFY med/lat
    6: 7,  # Lateral Meniscus          -> lateral_meniscus           << VERIFY med/lat
}

# Bộ nhãn UNION (không có bone vì SKM-TEA không gán xương). Dùng để ghi dataset.json.
UNION_LABEL_NAMES = {
    "background": 0,
    "femoral_cartilage": 2,
    "medial_tibial_cartilage": 4,
    "lateral_tibial_cartilage": 5,
    "medial_meniscus": 6,
    "lateral_meniscus": 7,
    "patellar_cartilage": 8,   # bỏ dòng này nếu LABEL_MAP[1]=0
}

print("OUT_ROOT =", OUT_ROOT)


## 1) Inspect — chạy TRƯỚC để biết cấu trúc thật

Xem: key nào trong `.h5` (echo1/echo2?), mask có **mấy nhãn** (6 = đã tách med/lat; 4 = còn gộp), shape & affine.

> Nếu mask chỉ có **4 nhãn** (1..4) → bản này **CHƯA tách** medial/lateral. Khi đó meniscus med/lat của bạn nên lấy từ **iMorphics**, hoặc phải tách hình học theo trục sagittal (không làm trong notebook này).


In [ ]:
import h5py, numpy as np, nibabel as nib

sample_id = "MTR_001"   # đổi sang ca bất kỳ nếu muốn
h5p = IMG_DIR / f"{sample_id}.h5"
mp  = MASK_DIR / f"{sample_id}.nii.gz"

print("=== H5:", h5p.name, "===")
with h5py.File(h5p, "r") as f:
    def _p(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name:24s} shape={obj.shape} dtype={obj.dtype}")
    f.visititems(_p)
    print("  top-level keys:", list(f.keys()))

print("\n=== MASK:", mp.name, "===")
mimg = nib.load(str(mp))
marr = np.asanyarray(mimg.dataobj)
print("  shape:", marr.shape, "ndim:", marr.ndim, "dtype:", marr.dtype)
if marr.ndim == 4:
    print("  -> mask 4D (one-hot). Sẽ đổi sang label-map bằng argmax khi convert.")
print("  unique labels:", np.unique(marr))
print("  affine:\n", np.array2string(mimg.affine, precision=3))


## 2) Hàm loader + remap (xử lý cả one-hot 4D)


In [ ]:
def load_qdess_rms(h5_path):
    # Doc echo1/echo2 -> RMS -> anh 1 kenh giong DESS
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())
        if "echo1" in keys and "echo2" in keys:
            e1 = np.asarray(f["echo1"]); e2 = np.asarray(f["echo2"])
        else:
            echo_keys = [k for k in keys if "echo" in k.lower()]
            if len(echo_keys) >= 2:
                e1 = np.asarray(f[echo_keys[0]]); e2 = np.asarray(f[echo_keys[1]])
            elif len(echo_keys) == 1 and np.asarray(f[echo_keys[0]]).shape[-1] == 2:
                arr = np.asarray(f[echo_keys[0]]); e1 = arr[..., 0]; e2 = arr[..., 1]
            else:
                raise KeyError(f"Khong tim thay echo1/echo2 trong {h5_path.name}. Keys={keys}")
    e1 = np.squeeze(np.abs(e1)).astype(np.float32)
    e2 = np.squeeze(np.abs(e2)).astype(np.float32)
    return np.sqrt((e1**2 + e2**2) / 2.0).astype(np.float32)

def to_labelmap(marr):
    # Neu mask 4D one-hot (H,W,D,C) theo class 1..C -> tra ve label-map 3D
    if marr.ndim == 4:
        fg  = marr.max(-1) > 0
        lab = marr.argmax(-1) + 1
        return np.where(fg, lab, 0).astype(np.int16)
    return marr.astype(np.int16)

def remap_labels(labelmap, mapping):
    out = np.zeros_like(labelmap, dtype=np.uint8)
    for src, dst in mapping.items():
        out[labelmap == src] = dst
    return out

def match_slice_axis(rms, ref_shape):
    # neu thu tu truc khac nhau -> hoan vi de bang shape mask
    if rms.shape != ref_shape and sorted(rms.shape) == sorted(ref_shape):
        for perm in [(0,1,2),(1,0,2),(0,2,1),(2,1,0),(1,2,0),(2,0,1)]:
            if np.transpose(rms, perm).shape == ref_shape:
                return np.ascontiguousarray(np.transpose(rms, perm))
    return rms

def orient_inplane(vol):
    # xoay/lat trong mat phang (axes 0,1) theo ORIENT_K / ORIENT_FLIP
    v = np.rot90(vol, ORIENT_K, axes=(0, 1))
    if ORIENT_FLIP:
        v = v[::-1]
    return np.ascontiguousarray(v)


## 3) Sanity-check medial vs lateral (bằng centroid)

Trong ảnh sagittal, **trục slice = trục medial–lateral**. Sụn chày *medial* và sụn chêm *medial* phải nằm **cùng phía** trên trục đó. Cell này in centroid từng nhãn gốc để bạn tự soi mapping có đúng không.


In [ ]:
lm = to_labelmap(np.asanyarray(nib.load(str(mp)).dataobj))
labs = [l for l in np.unique(lm) if l != 0]
cents = {int(l): np.argwhere(lm == l).mean(0) for l in labs}

# doan truc M-L = truc co phuong sai centroid lon nhat giua cac nhan
spread = np.array([[cents[l][a] for l in labs] for a in range(3)])
ml_axis = int(spread.var(1).argmax())
print(f"Truc medial-lateral (doan) = axis {ml_axis}\n")
print(f"{'label':>5} {'count':>9}   centroid(x,y,z)      pos tren truc M-L")
for l in labs:
    c = cents[l]; n = int((lm == l).sum())
    print(f"{l:5d} {n:9d}   ({c[0]:6.1f},{c[1]:6.1f},{c[2]:6.1f})   {c[ml_axis]:7.1f}")
print("\n>> Nhom cac nhan co pos gan nhau tren truc M-L = cung phia.")
print(">> Kiem tra: (Med Tibial, Med Meniscus) cung phia; (Lat Tibial, Lat Meniscus) cung phia.")
print(">> Neu nguoc, hoan doi cap medial/lateral trong LABEL_MAP.")


## 3.5) Hiệu chỉnh HƯỚNG ảnh ↔ mask (quan trọng)

`.h5` (ảnh) và `.nii.gz` (mask) có thể lưu mảng theo thứ tự trục khác nhau → ảnh bị **xoay 90°** so với mask. Cell dưới vẽ **8 biến thể xoay/lật** của ảnh đè lên mask + chấm điểm (cường độ dưới mask; hướng đúng → sụn sáng → điểm cao).

**Chọn ô mà mask (màu) nằm ĐÚNG trên sụn/xương**, rồi điền `ORIENT_K`, `ORIENT_FLIP` vào **cell 0** và chạy lại từ cell 0.


In [ ]:
import matplotlib.pyplot as plt

lm0  = to_labelmap(np.asanyarray(nib.load(str(MASK_DIR / f"{sample_id}.nii.gz")).dataobj))
rms0 = match_slice_axis(load_qdess_rms(IMG_DIR / f"{sample_id}.h5"), lm0.shape)
img_n = (rms0 - rms0.min()) / (np.ptp(rms0) + 1e-6)
z = int(np.argmax([(lm0[:, :, i] > 0).sum() for i in range(lm0.shape[2])]))

combos = [(k, f) for f in (False, True) for k in range(4)]
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
best = None
for i, (k, flip) in enumerate(combos):
    vi = np.rot90(img_n, k, axes=(0, 1))
    if flip: vi = vi[::-1]
    if vi.shape != lm0.shape:
        continue
    m = lm0[:, :, z] > 0
    sc = float(vi[:, :, z][m].mean()) if m.any() else 0.0
    ax = axes[i // 4, i % 4]
    ax.imshow(vi[:, :, z].T, cmap="gray", origin="lower")
    mm = np.ma.masked_where(lm0[:, :, z] == 0, lm0[:, :, z])
    ax.imshow(mm.T, cmap="autumn", alpha=0.6, origin="lower")
    ax.set_title(f"ORIENT_K={k} FLIP={flip}  score={sc:.3f}")
    ax.axis("off")
    if best is None or sc > best[0]:
        best = (sc, k, flip)
plt.suptitle("Chon o co MASK nam dung sun/xuong. (score cao = kha nang dung)")
plt.tight_layout(); plt.show()
print(f">> Goi y: ORIENT_K = {best[1]} , ORIENT_FLIP = {best[2]}  (score={best[0]:.3f})")
print(">> Dien vao CELL 0 roi chay lai tu cell 0. Sau do xem lai QC o cell 4.")


## 4) Convert thử 1 ca + QC overlay (BẮT BUỘC xem mắt)

Nếu mask (màu) nằm đúng trên sụn/meniscus của ảnh xám → geometry OK. Nếu lệch/xoay → có transpose/flip trong mặt phẳng, báo lại để chỉnh.


In [ ]:
import matplotlib.pyplot as plt

def convert_one(case_id, save=False):
    h5p = IMG_DIR / f"{case_id}.h5"
    mp  = MASK_DIR / f"{case_id}.nii.gz"
    rms = load_qdess_rms(h5p)
    mimg = nib.load(str(mp))
    lm = to_labelmap(np.asanyarray(mimg.dataobj))
    affine = mimg.affine
    rms = match_slice_axis(rms, lm.shape)   # 1) khop truc slice
    rms = orient_inplane(rms)               # 2) xoay/lat trong mat phang cho khop mask
    if rms.shape != lm.shape:
        raise ValueError(f"{case_id}: shape anh {rms.shape} != mask {lm.shape}")
    lbl = remap_labels(lm, LABEL_MAP)
    if save:
        nib.save(nib.Nifti1Image(rms, affine),               str(IMAGES_TR / f"{case_id}_0000.nii.gz"))
        nib.save(nib.Nifti1Image(lbl.astype(np.uint8), affine), str(LABELS_TR / f"{case_id}.nii.gz"))
    return rms, lbl, affine

import matplotlib.colors as mcolors
from matplotlib.patches import Patch

rms, lbl, aff = convert_one(sample_id, save=False)
print("image", rms.shape, rms.dtype)

UNION_NAMES_BY_ID = {v: k for k, v in UNION_LABEL_NAMES.items()}
FIXED_COLORS = {2: "#1f77b4", 4: "#2ca02c", 5: "#bcbd22",
                6: "#ff7f0e", 7: "#d62728", 8: "#17becf"}   # mau co dinh theo union-id

present = [int(l) for l in np.unique(lbl) if l != 0]
print("\nNhan co mat | ten class | so voxel | centroid(x,y,z):")
for l in present:
    idx = np.argwhere(lbl == l); c = idx.mean(0)
    print(f"  {l}  {UNION_NAMES_BY_ID.get(l,'?'):24s} n={len(idx):7d}  ({c[0]:.0f},{c[1]:.0f},{c[2]:.0f})")

def overlay2d(ax, img2d, lbl2d):
    ax.imshow(img2d.T, cmap="gray", origin="lower")
    rgba = np.zeros(lbl2d.shape + (4,))
    for l in present:
        m = lbl2d == l
        if m.any():
            rgba[m, :3] = mcolors.to_rgb(FIXED_COLORS.get(l, "#ffffff")); rgba[m, 3] = 0.6
    ax.imshow(np.transpose(rgba, (1, 0, 2)), origin="lower")
    ax.axis("off")

counts = [(lbl[:, :, z] > 0).sum() for z in range(lbl.shape[2])]
zs = sorted(int(z) for z in np.argsort(counts)[::-1][:3])   # 3 slice nhieu nhan nhat
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, z in zip(axes, zs):
    overlay2d(ax, rms[:, :, z], lbl[:, :, z]); ax.set_title(f"slice {z}")
handles = [Patch(color=FIXED_COLORS.get(l, "#fff"),
                 label=f"{l} {UNION_NAMES_BY_ID.get(l,'?')}") for l in present]
fig.legend(handles=handles, loc="lower center", ncol=min(len(present), 4), fontsize=9)
fig.suptitle(f"{sample_id} — QC overlay (mau ↔ class trong chu thich)")
plt.tight_layout(rect=[0, 0.06, 1, 1]); plt.show()


## 5) Convert TOÀN BỘ → lưu vào Drive (convert 1 lần, dùng mãi)

Output ghi thẳng vào `OUT_ROOT` trên Drive nên **tồn tại vĩnh viễn**. Ca nào đã có đủ ảnh+nhãn sẽ được **bỏ qua** → chạy lại (hoặc chạy tiếp sau khi bị ngắt) rất nhanh. Đặt `FORCE=True` nếu muốn convert lại từ đầu.


In [ ]:
from tqdm.auto import tqdm

FORCE = False   # True = convert lai tat ca (ghi de)

case_ids = sorted(p.stem for p in IMG_DIR.glob("*.h5"))
case_ids = [c for c in case_ids if (MASK_DIR / f"{c}.nii.gz").exists()]
print(f"So ca co ca anh + mask: {len(case_ids)}")

ok, fail, skip = [], [], []
for c in tqdm(case_ids):
    img_out = IMAGES_TR / f"{c}_0000.nii.gz"
    lbl_out = LABELS_TR / f"{c}.nii.gz"
    if not FORCE and img_out.exists() and lbl_out.exists():
        skip.append(c); ok.append(c); continue     # da convert -> bo qua
    try:
        convert_one(c, save=True)
        ok.append(c)
    except Exception as e:
        fail.append((c, str(e)))
        print("LOI", c, "->", e)

print(f"\nXong. Tong OK={len(ok)}  (moi convert={len(ok)-len(skip)}, skip san co={len(skip)})  FAIL={len(fail)}")
for c, e in fail:
    print("   ", c, e)


## 6) Ghi `dataset.json`

> **Lưu ý về nnU-Net:** folder này dùng **bộ nhãn UNION** (2,4,5,6,7,8 — thiếu bone 1,3) nên **chỉ dùng làm nguồn để MERGE** với OAI-ZIB (cung cấp 1,3) rồi mới `plan_and_preprocess` trên dataset gộp.
>
> Nếu muốn train **model SKM-TEA riêng** (chỉ sụn+meniscus), hãy sửa `LABEL_MAP` sang **1..6 liên tiếp** và `UNION_LABEL_NAMES` tương ứng (nnU-Net yêu cầu nhãn liên tiếp bắt đầu từ 0).


In [ ]:
import json

used = set()
for c in ok:
    used |= set(np.unique(np.asanyarray(nib.load(str(LABELS_TR / f"{c}.nii.gz")).dataobj)).tolist())
labels = {k: v for k, v in sorted(UNION_LABEL_NAMES.items(), key=lambda kv: kv[1]) if v == 0 or v in used}

dataset = {
    "channel_names": {"0": CHANNEL_NAME},
    "labels": labels,
    "numTraining": len(ok),
    "file_ending": ".nii.gz",
    "overwrite_image_reader_writer": "SimpleITKIO",
    "description": "SKM-TEA qDESS (RMS echo1/echo2) -> nnU-Net; nhan theo bo UNION (khong co bone).",
}
with open(OUT_ROOT / "dataset.json", "w") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print(json.dumps(dataset, indent=2, ensure_ascii=False))


## 7) (Tùy chọn) Đọc split chính thức của SKM-TEA

Để tránh leakage khi báo cáo, có thể tôn trọng split train/val/test gốc. Cell này đọc thử và trích các `MTR_xxx`.


In [ ]:
import json
for name in ["train", "val", "test"]:
    p = ANN_DIR / f"{name}.json"
    if not p.exists():
        print(name, "-> khong thay", p); continue
    d = json.load(open(p))
    # thu vai schema pho bien cua SKM-TEA (COCO-like: 'images' voi 'file_name'/'scan_id')
    ids = set()
    if isinstance(d, dict) and "images" in d:
        for im in d["images"]:
            v = im.get("scan_id") or im.get("file_name") or ""
            for tok in str(v).replace("/", "_").split("_"):
                if tok.startswith("MTR"):
                    ids.add(tok)
            if str(v).startswith("MTR"):
                ids.add(str(v).split(".")[0])
    print(f"{name:5s}: {len(ids)} scans  (vd: {sorted(ids)[:5]})")
    print("   top-level keys:", list(d.keys()) if isinstance(d, dict) else type(d))
